# CNN full training (all 3 CV folds)

Full multi-epoch training run, replacing notebook 06's earlier one-epoch speed test.
Same dataset/CV-split harness as notebook 05 (constant baseline) and the earlier speed
test, now actually trained to see if it beats the trivial baseline. Still an
**exploratory** run (`.CLAUDE.md` §0) — the working tree isn't committed, so these
numbers aren't decision-grade; this is about validating the training loop and getting a
first read on whether the model learns anything real.

Device selection (CUDA / Apple MPS / CPU) and all paths are auto-detected — portable to
a Linux CUDA machine without edits.

In [ ]:
import sys
import pickle
import time
from pathlib import Path

# Auto-detect repo root: search upward from cwd for pyproject.toml -- robust to
# whatever directory Jupyter was launched from (local VS Code, remote Linux server, etc.)
def _find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    return p  # fallback: no marker found, use cwd as-is

REPO = _find_repo_root()
sys.path.insert(0, str(REPO / 'src'))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

from preprocessing import loto_cv_splits, HELD_OUT_TEST_TRACK

DATASETS_DIR = REPO / 'processed_data' / 'datasets'
run_dir = sorted(DATASETS_DIR.iterdir())[-1]
print('using processed data run:', run_dir.name)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('using device:', device)

torch.manual_seed(0)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(0)

## Dataset, metric, model

Same as notebooks 05/06 (constant baseline / speed test) — unchanged, so results are
directly comparable to what's already been reported.

In [ ]:
TARGET_KEY = 'width_mean_mm'
THERMAL_MEAN, THERMAL_STD = 1750.0, 750.0

class TrackSampleDataset(Dataset):
    def __init__(self, run_dir, track_ids, target_key=TARGET_KEY):
        self.rows = []
        for track_id in track_ids:
            with open(Path(run_dir) / f'track_{track_id}_samples.pkl', 'rb') as f:
                track_rows = pickle.load(f)
            self.rows.extend(r for r in track_rows if r['valid'])
        self.target_key = target_key

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        thermal = torch.from_numpy(row['thermal_window'].copy())
        target = torch.tensor(row[self.target_key], dtype=torch.float32)
        return thermal, target

    def targets(self):
        return np.array([r[self.target_key] for r in self.rows], dtype=np.float64)


def mae_mm(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(np.abs(y_true - y_pred)))


class SimpleCNN(nn.Module):
    def __init__(self, in_channels=11):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=5, stride=2, padding=2), nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, kernel_size=5, stride=2, padding=2), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(64, 1)

    def forward(self, x):
        feat = self.features(x).flatten(1)
        return self.head(feat).squeeze(-1)


# constant-baseline reference numbers, from notebook 05 (for comparison below)
BASELINE_VAL_MAE = {8: 0.2376, 10: 0.1501, 14: 0.2572}

## Training loop

`N_EPOCHS = 20`, Adam (`lr=1e-3`), L1 loss (matches the MAE metric). One run per Task-3
CV fold. Val is evaluated every epoch (no gradient updates on it — same "fit-on-train-
only" principle as the baseline). Best-val-epoch weights are kept per fold, not just the
final epoch, since a small model on ~700 samples can start overfitting well before
epoch 20.

In [ ]:
N_EPOCHS = 20
BATCH_SIZE = 16
LR = 1e-3

# pin_memory + worker processes materially speed up data loading on a CUDA
# machine (overlaps host->device transfer with compute); neither helps on MPS/CPU.
DATALOADER_KWARGS = dict(pin_memory=True, num_workers=4) if device.type == 'cuda' else dict()
print('DataLoader kwargs for this device:', DATALOADER_KWARGS)

def sync():
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elif device.type == 'mps':
        torch.mps.synchronize()

def run_epoch(model, loader, optimizer, loss_fn, train):
    model.train(train)
    all_true, all_pred = [], []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for thermal, target in loader:
            thermal = ((thermal - THERMAL_MEAN) / THERMAL_STD).to(device)
            target = target.to(device)
            if train:
                optimizer.zero_grad()
            pred = model(thermal)
            loss = loss_fn(pred, target)
            if train:
                loss.backward()
                optimizer.step()
            all_true.append(target.detach().cpu().numpy())
            all_pred.append(pred.detach().cpu().numpy())
    return mae_mm(np.concatenate(all_true), np.concatenate(all_pred))


fold_results = []
t_total_start = time.time()
for fold_i, (train_tracks, val_track) in enumerate(loto_cv_splits()):
    print(f'\n=== fold {fold_i+1}/3: train={train_tracks} val={val_track} ===')
    train_ds = TrackSampleDataset(run_dir, train_tracks)
    val_ds = TrackSampleDataset(run_dir, [val_track])
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, **DATALOADER_KWARGS)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **DATALOADER_KWARGS)

    model = SimpleCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.L1Loss()

    history = {'train_mae': [], 'val_mae': []}
    best_val_mae = float('inf')
    best_epoch = -1
    t_fold_start = time.time()
    for epoch in range(N_EPOCHS):
        train_mae = run_epoch(model, train_loader, optimizer, loss_fn, train=True)
        val_mae = run_epoch(model, val_loader, optimizer, loss_fn, train=False)
        history['train_mae'].append(train_mae)
        history['val_mae'].append(val_mae)
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_epoch = epoch
        print(f'  epoch {epoch+1:2d}/{N_EPOCHS}: train_MAE={train_mae:.4f}mm  val_MAE={val_mae:.4f}mm')
    fold_time = time.time() - t_fold_start

    fold_results.append(dict(
        fold=fold_i + 1, train_tracks=train_tracks, val_track=val_track,
        history=history, best_val_mae=best_val_mae, best_epoch=best_epoch,
        final_val_mae=history['val_mae'][-1], fold_time_sec=fold_time,
    ))
    print(f'  fold time: {fold_time:.1f}s, best val_MAE={best_val_mae:.4f}mm at epoch {best_epoch+1}, '
          f'baseline val_MAE={BASELINE_VAL_MAE[val_track]:.4f}mm')

total_time = time.time() - t_total_start
print(f'\ntotal training time (3 folds x {N_EPOCHS} epochs): {total_time:.1f}s')

## Loss curves per fold

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, r in zip(axes, fold_results):
    ax.plot(range(1, N_EPOCHS + 1), r['history']['train_mae'], label='train MAE', color='#3366cc')
    ax.plot(range(1, N_EPOCHS + 1), r['history']['val_mae'], label='val MAE', color='#cc3333')
    ax.axhline(BASELINE_VAL_MAE[r['val_track']], color='#999999', ls='--', lw=1, label='constant baseline val MAE')
    ax.axvline(r['best_epoch'] + 1, color='#cc3333', ls=':', lw=1, alpha=0.6)
    ax.set_title(f"fold {r['fold']}: train={r['train_tracks']} val={r['val_track']}")
    ax.set_xlabel('epoch')
    ax.legend(fontsize=8)
axes[0].set_ylabel('MAE (mm)')
plt.suptitle('CNN training vs. constant baseline, per CV fold')
plt.tight_layout()
plt.show()

## Summary: CNN (best epoch) vs. constant baseline

In [ ]:
summary = pd.DataFrame([
    dict(fold=r['fold'], val_track=r['val_track'], n_train=len(TrackSampleDataset(run_dir, r['train_tracks'])),
         n_val=len(TrackSampleDataset(run_dir, [r['val_track']])),
         baseline_val_mae=BASELINE_VAL_MAE[r['val_track']],
         cnn_best_val_mae=r['best_val_mae'], cnn_best_epoch=r['best_epoch'] + 1,
         cnn_final_val_mae=r['final_val_mae'])
    for r in fold_results
])
summary['improvement_vs_baseline_pct'] = 100 * (summary['baseline_val_mae'] - summary['cnn_best_val_mae']) / summary['baseline_val_mae']
summary

In [ ]:
print(f"mean baseline val MAE across folds: {summary['baseline_val_mae'].mean():.4f}mm")
print(f"mean CNN best val MAE across folds:      {summary['cnn_best_val_mae'].mean():.4f}mm")
print(f"mean improvement: {summary['improvement_vs_baseline_pct'].mean():.1f}%")

## Notes

- **Still exploratory** (`.CLAUDE.md` §0): working tree isn't committed, so none of
  these numbers are decision-grade yet — this is a first read on whether the model
  learns real signal, not a result to cite.
- Best-val-epoch weights are reported per fold (not necessarily the final epoch) —
  check the loss-curve plots above for whether/when val MAE starts rising again
  (overfitting) after its minimum.
- Only `width_mean_mm` (single scalar target) and only the thermal modality are used
  here — SEM context and the other four target stats (boundary positions, edge
  roughness) aren't used by this model yet.
- Track 21 (held-out test) is still untouched, per `.CLAUDE.md` §3.